# 19.10 如何把成品帶到較小的電腦？


一份大模型可用較少位元近似儲存，也可另外訓練較小學生。前者是量化，改變同一結構的數值；後者是蒸餾，學生結構先變小，再學教師訊號。兩條路各有用途，不必全部套在共同成品上才算完成。

先看一層4×8權重，不牽涉整個助手：


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch
from torch import nn

from tiny_perceptron.quantization import QuantizedLinear

torch.manual_seed(42)
layer = nn.Linear(8, 4)
compressed = QuantizedLinear(layer, bits=4)
x = torch.ones(1, 8)
with torch.no_grad():
    error = (layer(x) - compressed(x)).abs().max().item()
print("FP32權重加bias bytes", sum(p.numel() * p.element_size() for p in layer.parameters()))
print("打包權重、刻度與bias bytes", compressed.storage_bytes(), "最大輸出差", error)

FP32 的32個權重加4個 bias 共144 bytes。4-bit 打包權重16 bytes，加上刻度和 bias 各16，這個物件的張量儲存共48 bytes。原輸入是八個1；最大輸出差只針對這一筆，不是整份模型答錯率。這個可讀實作用浮點反量化計算，所以檔案小不直接代表執行更快。

完整助手壓縮後要真的存檔、重載，再核對指令、指定框、語音、工具參數與 EOS。檔案大小、載入後張量與執行峯值各量各的；某題原本已錯，壓縮後仍錯而總分相同，不等於輸出完全一致。

若改成 Dense 學生，則說明它與教師的架構、資料與訓練預算。教師固定，學生可同時學標準答案 CE 和教師分佈；舊配方用 `0.5 CE + 0.5 T² KL(p_T||q_T)`，T=2，內部已乘 T²，不再乘一次。它也要與同尺寸只學標準答案的學生比，才能分清縮小和教師訊號的影響。新成品是否需要這條部署路線，由資源與能力取捨決定。

<details>
<summary>補充：舊合成任務的量化與學生比較</summary>

推薦joint的兩種PTQ已真的存檔、重新載入，再重跑同一份90題最後檢查。下面的檔案bytes取自部署測試當時的檔案，包含容器與metadata，也就是描述模型結構、來源等附加資訊；後續公開版若移除內部路徑等資訊，檔案大小與指紋可能改變，公開下載要核對19.11的正式清單。

| 推薦joint版本 | 測試當時檔案bytes | 儲存張量bytes | 最後整題正確 |
| --- | --- | --- | --- |
| FP32來源檔 | 1,345,023 | 1,312,512 | 78／90 |
| PTQ 4-bit | 275,381 | 248,864 | 78／90 |
| PTQ 8-bit | 429,365 | 402,720 | 78／90 |

三列各任務的正確數也相同，但4-bit並沒有逐token相同：12個完整計算迴圈裡，原本「請算1加0」讀回後答`111`，4-bit改成`11`，兩個都錯，因此總分不變。整份90題共有1題的生成ID序列改變；8-bit在這90題的生成ID則沒有觀察到差異。這與19.9同一權重的快取一致對照是不同實驗，不能互相替代。

DPO比較分支的FP32／4-bit／8-bit同樣都是78／90；其4-bit有2題生成ID改變，8-bit沒有觀察到差異。因此「同分」只說明這份題庫的評分結果，不等於整個模型數字、所有回答或其他輸入都相同。本輪保留原本就錯的形狀與工具讀回案例，沒有把同分說成量化修好了它們。

儲存規格、來源指紋與量化欄位見[部署實報](https://github.com/birdhackor/tiny-perceptron-vlm/blob/1df335318bda03fd771807f66976953231d5a00b/docs/course-experiments/results/capstone_deployment.json)；逐題可並排看[joint FP32](https://github.com/birdhackor/tiny-perceptron-vlm/blob/1df335318bda03fd771807f66976953231d5a00b/docs/course-experiments/capstone-evidence/deployment/test-joint.json)、[joint 4-bit](https://github.com/birdhackor/tiny-perceptron-vlm/blob/1df335318bda03fd771807f66976953231d5a00b/docs/course-experiments/capstone-evidence/deployment/test-joint-ptq4.json)與[joint 8-bit](https://github.com/birdhackor/tiny-perceptron-vlm/blob/1df335318bda03fd771807f66976953231d5a00b/docs/course-experiments/capstone-evidence/deployment/test-joint-ptq8.json)。這些檔案載入後仍還原FP32，不代表執行記憶體也縮為八分之一。

學生比較分兩支：CE只學標準答案的交叉熵；KD則同時學標準答案與教師的候選分佈，配方為`0.5 CE + 0.5 T² KL(p_T||q_T)`，溫度T=2；p_T與q_T分別是教師與學生在此溫度下的候選分佈。`distillation_loss`內部已乘T²，呼叫處不再乘一次。KL量兩份分佈的差，詳細直覺見[18.8](https://birdhackor.github.io/tiny-perceptron-vlm/18.8.html)，混合示範與教師訊號見[18.9](https://birdhackor.github.io/tiny-perceptron-vlm/18.9.html)。教師保持固定，只有學生更新。

兩支複製同一份隨機Dense起點，固定相同的抽樣種子、批次規則與350次更新，各累計145,163個有效回答位置。這控制了配方；報告沒有逐批保存GPU取樣清單，不能稱做過逐批實錄核對。KD還需要固定教師的前向計算，訓練成本因此不能只用學生參數數量估計；完整計時與來源核對保留在下方實報。

驗證各為59／84，最後整題正確卻是CE 62／90、KD 61／90。這次蒸餾沒有勝過同尺寸、同起點的CE學生。真正差一題的例子是「1+8等於多少？」：CE請求`TOOL:calculator:1+8`，工具回9，最後答9；KD卻請求`TOOL:calculator:1+9`，工具因而回10，模型最後又答11。請求參數和讀回兩層都錯了，不能只看最後總分來說「大致都保留下來」。

下面各分項都是同一份90題考卷的子集；成對數字先列CE、再列KD。[完整計算迴圈](https://birdhackor.github.io/tiny-perceptron-vlm/19.7.html)包含模型請求、工具執行與模型讀回答案，兩支只有1／12與0／12。獨立回填是直接提供原題與工具結果，省掉先前的請求步驟，兩支都只有1／6。照抄要求原樣重現指定數字，兩支都是0／3；形狀要求辨識圖片是方形或圓形，兩支都是0／9。這些[成品題型](https://birdhackor.github.io/tiny-perceptron-vlm/19.1.html)各自考不同能力，不能把分項再加進90題總數。

| 小Dense交付版本 | 測試當時檔案bytes | 儲存張量bytes | 最後整題正確 |
| --- | --- | --- | --- |
| CE學生FP32 | 342,451 | 319,680 | 62／90 |
| KD學生FP32 | 342,451 | 319,680 | 61／90 |
| KD學生4-bit | 106,229 | 91,680 | 61／90 |

KD的4-bit也真的存檔、重載；它有6／90題的動作或最後生成ID改變，均發生在原本已錯的題目，總分才仍是61／90。小模型與打包確實減少儲存，這個固定350步支線卻沒有完整保留教師能力。本輪也沒有量學生的推論速度或生成記憶體峯值，所以不能從參數較少推出一個速度數字；這不是對所有Dense、所有蒸餾配方的結論。

所有條件與分數見[學生實報](https://github.com/birdhackor/tiny-perceptron-vlm/blob/1df335318bda03fd771807f66976953231d5a00b/docs/course-experiments/results/capstone_student.json)，同題可並排看[CE逐題檔](https://github.com/birdhackor/tiny-perceptron-vlm/blob/1df335318bda03fd771807f66976953231d5a00b/docs/course-experiments/capstone-evidence/student/test-ce.json)、[KD逐題檔](https://github.com/birdhackor/tiny-perceptron-vlm/blob/1df335318bda03fd771807f66976953231d5a00b/docs/course-experiments/capstone-evidence/student/test-kd.json)與[KD 4-bit逐題檔](https://github.com/birdhackor/tiny-perceptron-vlm/blob/1df335318bda03fd771807f66976953231d5a00b/docs/course-experiments/capstone-evidence/student/test-kd-ptq4.json)。若你想重做壓縮研究，先保留這組比較與失敗，再為新配方另切驗證／最後考卷；不要看這次最後題目後挑比較好的一次重跑冒充原實驗。

</details>
